In [ ]:
!pip install transformers datasets torchaudio accelerate matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

from datasets import load_dataset, Audio, DatasetDict
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from scipy.interpolate import interp1d
from tqdm import tqdm

In [ ]:
TARGET_IDX = 35

### Φόρτωση αρχικού και εκπαιδευμένου μοντέλου.

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"

# Μετά το finetuning
model_after = AutoModelForAudioClassification.from_pretrained(repo_id)

feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

model_after.eval()

#Ίδιες ετικέτες με το finetuned μοντέλο, ώστε τα δύο μοντέλα να είναι συγκρίσιμα
label2id = model_after.config.label2id
id2label = model_after.config.id2label
num_labels = model_after.config.num_labels

#Πριν το finetuning: προεκπαιδευμένος encoder + τυχαία αρχικοποιημένο classification head
model_before = AutoModelForAudioClassification.from_pretrained(
    "facebook/data2vec-audio-base-960h",
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

model_before.eval()


### Φόρτωση dataset (speaker-independent)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

In [ ]:
def resample_attribution(attr, target_frames=50):
    attr = np.abs(attr)
    x_original = np.linspace(0, 1, len(attr))
    x_target = np.linspace(0, 1, target_frames)
    f = interp1d(x_original, attr, kind='linear')
    resampled = f(x_target)
    resampled = resampled / (resampled.sum() + 1e-10)
    return resampled

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
sample = test_dataset[TARGET_IDX]
audio = sample["audio"]["array"]

### Συνάρτηση υπολογισμού saliency

In [ ]:
def compute_saliency(model, audio, target_class=None):
    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
    input_values = inputs["input_values"]
    input_values.requires_grad = True

    outputs = model(input_values=input_values)
    logits = outputs.logits

    if target_class is None:
        target_class = logits.argmax(dim=-1).item()

    model.zero_grad()
    logits[0, target_class].backward()

    saliency = input_values.grad.abs().squeeze().detach().numpy()
    audio_np = input_values.squeeze().detach().numpy()

    return saliency, audio_np, target_class

### Υπολογισμός saliency και για τα δύο μοντέλα

In [ ]:
saliency_after, audio_np, predicted_class = compute_saliency(model_after, audio, target_class=None)
print("Predicted emotion (μετά το finetuning):", model_after.config.id2label[predicted_class])

saliency_before, _, _ = compute_saliency(model_before, audio, target_class=predicted_class)

#Κοινή κανονικοποίηση
shared_max = saliency_after.max()

saliency_after_norm = saliency_after / shared_max
saliency_before_norm = saliency_before / shared_max

### Οπτικοποίηση: ξεχωριστά subplots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

axes[0].plot(audio_np, alpha=0.5, color="gray", label="Audio Signal")
axes[0].plot(saliency_before_norm, color="tab:blue", label="Saliency")
axes[0].set_title("Πριν το finetuning")
axes[0].set_ylabel("Importance")
axes[0].legend()

axes[1].plot(audio_np, alpha=0.5, color="gray", label="Audio Signal")
axes[1].plot(saliency_after_norm, color="tab:orange", label="Saliency")
axes[1].set_title("Μετά το finetuning")
axes[1].set_xlabel("Audio Samples")
axes[1].set_ylabel("Importance")
axes[1].legend()

plt.tight_layout()
plt.show()

### Ποσοτική σύγκριση των δύο κατανομών

In [ ]:
correlation = np.corrcoef(saliency_before, saliency_after)[0, 1]

print(f"Συσχέτιση (Pearson) saliency πριν/μετά: {correlation:.3f}")

# Πόσο μετατοπίστηκε η "μάζα" του saliency (L1 απόσταση κατανομών, κανονικοποιημένων ώστε να αθροίζουν σε 1)
dist_before = saliency_before / saliency_before.sum()
dist_after = saliency_after / saliency_after.sum()

l1_shift = np.abs(dist_before - dist_after).sum() / 2  # 0 = ίδια κατανομή, 1 = τελείως διαφορετική

print(f"Μετατόπιση κατανομής saliency (L1, 0-1): {l1_shift:.3f}")


In [ ]:
resampled_saliency = resample_attribution(saliency_after, target_frames=50)

np.save(f"{save_dir}/Saliency_idx{TARGET_IDX}.npy", resampled_saliency)
print(f"Αποθηκεύτηκε: Saliency_idx{TARGET_IDX}.npy, shape={resampled_saliency.shape}")

print(os.listdir(save_dir))